# Evaluate multiple checkpoints on the PathVQA test set

Notebook này dành cho Kaggle. Hãy bật GPU, gắn PathVQA (thư mục có `train.json`, `test.json`, `answer_list.json`, `images/`) và các checkpoint vào notebook. Sửa `CHECKPOINT_PATHS` nếu muốn chỉ định chính xác checkpoint; để danh sách rỗng nếu muốn tự động tìm.

Mỗi checkpoint được chạy độc lập bằng `train_vqa.py --evaluate`, sau đó chấm bằng `pathvqa_eval.py`. Kết quả tổng hợp được lưu dưới dạng JSON và CSV.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')

# Đặt đường dẫn cụ thể nếu đã biết; giữ None để notebook tự tìm.
PATHVQA_ROOT = None
PATHVQA_SEARCH_ROOTS = [Path('/kaggle/working/pathvqa'), Path('/kaggle/input')]

# Có thể truyền một hoặc nhiều checkpoint tại đây.
CHECKPOINT_PATHS = [
    # '/kaggle/input/my-checkpoints/checkpoint_09.pth',
    # '/kaggle/input/my-checkpoints/checkpoint_19.pth',
]
CHECKPOINT_SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
CHECKPOINT_GLOB = 'checkpoint_*.pth'
MAX_CHECKPOINTS = None

OUTPUT_ROOT = Path('/kaggle/working/pathvqa_eval')
CONFIG_PATH = REPO_DIR / 'configs/pathvqa_eval_kaggle.yaml'
CONFIG_ARG = 'configs/pathvqa_eval_kaggle.yaml'
MAX_GPUS = 2
BATCH_SIZE_TEST = 4
K_TEST = 128
INFERENCE = 'rank'
RERUN_EXISTING = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import sys

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Using existing repository: {REPO_DIR}')

packages = [
    'omegaconf==2.3.0',
    'hydra-core==1.3.2',
    'timm==0.4.12',
    'fairscale==0.4.13',
    'transformers==4.36.1',
    'pandas',
    'pyyaml',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print(f'Repository ready: {REPO_DIR}')

In [ ]:
import json

def is_pathvqa_root(path):
    path = Path(path)
    required = ['train.json', 'test.json', 'answer_list.json']
    return all((path / name).is_file() for name in required) and (path / 'images/test').is_dir()

def find_pathvqa_root(explicit_root, search_roots):
    if explicit_root is not None:
        candidate = Path(explicit_root).expanduser().resolve()
        if not is_pathvqa_root(candidate):
            raise FileNotFoundError(f'PATHVQA_ROOT is invalid: {candidate}')
        return candidate

    checked = set()
    for search_root in search_roots:
        search_root = Path(search_root)
        if not search_root.exists():
            continue
        direct_candidates = [search_root, search_root / 'pathvqa', search_root / 'PathVQA']
        for candidate in direct_candidates:
            resolved = candidate.resolve()
            if resolved not in checked and is_pathvqa_root(resolved):
                return resolved
            checked.add(resolved)
        for test_json in search_root.rglob('test.json'):
            candidate = test_json.parent.resolve()
            if candidate not in checked and is_pathvqa_root(candidate):
                return candidate
            checked.add(candidate)
    raise FileNotFoundError(
        'Could not find PathVQA. Attach the dataset or set PATHVQA_ROOT explicitly.'
    )

pathvqa_root = find_pathvqa_root(PATHVQA_ROOT, PATHVQA_SEARCH_ROOTS)
with (pathvqa_root / 'test.json').open(encoding='utf-8') as file:
    test_records = json.load(file)
if not test_records:
    raise ValueError('PathVQA test.json is empty.')

missing_images = []
for record in test_records:
    image_path = record.get('image') or record.get('image_path')
    if image_path and not (pathvqa_root / 'images' / image_path).is_file():
        missing_images.append(image_path)
        if len(missing_images) >= 10:
            break
if missing_images:
    raise FileNotFoundError(f'Missing PathVQA test images, examples: {missing_images}')

print(f'PathVQA root: {pathvqa_root}')
print(f'Test examples: {len(test_records):,}')

In [ ]:
import re

def checkpoint_label(path):
    raw = f'{path.parent.name}_{path.stem}'
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', raw).strip('_.-') or 'checkpoint'

if CHECKPOINT_PATHS:
    discovered_checkpoints = [Path(path).expanduser().resolve() for path in CHECKPOINT_PATHS]
else:
    discovered_checkpoints = []
    for search_root in CHECKPOINT_SEARCH_ROOTS:
        search_root = Path(search_root)
        if search_root.exists():
            discovered_checkpoints.extend(search_root.rglob(CHECKPOINT_GLOB))
    discovered_checkpoints = sorted({path.resolve() for path in discovered_checkpoints})

valid_checkpoints = []
for checkpoint in discovered_checkpoints:
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint}')
    if checkpoint.stat().st_size <= 1_000_000:
        print(f'Skipping suspiciously small checkpoint: {checkpoint}')
        continue
    valid_checkpoints.append(checkpoint)

if MAX_CHECKPOINTS is not None:
    valid_checkpoints = valid_checkpoints[:MAX_CHECKPOINTS]
if not valid_checkpoints:
    raise FileNotFoundError('No valid checkpoints found. Fill CHECKPOINT_PATHS or adjust the search roots.')

checkpoint_paths = {}
for checkpoint in valid_checkpoints:
    base_label = checkpoint_label(checkpoint)
    label = base_label
    suffix = 2
    while label in checkpoint_paths:
        label = f'{base_label}_{suffix}'
        suffix += 1
    checkpoint_paths[label] = checkpoint

print(f'Checkpoints to evaluate: {len(checkpoint_paths)}')
for label, checkpoint in checkpoint_paths.items():
    print(f'  {label}: {checkpoint}')

In [ ]:
import yaml

eval_config = {
    'ann_root': str(pathvqa_root),
    'vqa_root': str(pathvqa_root / 'images'),
    'train_files': ['train'],
    'dataset_name': 'pathvqa',
    'truncate_train_dataset_to': None,
    'pretrained': str(next(iter(checkpoint_paths.values()))),
    'vit': 'base',
    'batch_size_train': 1,
    'batch_size_test': BATCH_SIZE_TEST,
    'vit_grad_ckpt': False,
    'vit_ckpt_layer': 0,
    'init_lr': 2e-5,
    'image_size': 480,
    'k_test': K_TEST,
    'inference': INFERENCE,
    'weight_decay': 0.05,
    'min_lr': 0,
    'max_epoch': 1,
    'torch_home': '/kaggle/working/torch_home',
    'wandb': False,
    'save_last_only': True,
}
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open('w', encoding='utf-8') as file:
    yaml.safe_dump(eval_config, file, sort_keys=False)
print(CONFIG_PATH.read_text(encoding='utf-8'))

In [ ]:
import torch

available_gpus = torch.cuda.device_count()
if available_gpus < 1:
    raise RuntimeError('No CUDA GPU detected. Enable a GPU accelerator in Kaggle settings.')
gpu_count = min(MAX_GPUS, available_gpus)
print(f'Using {gpu_count} of {available_gpus} available GPU(s)')
for index in range(available_gpus):
    print(f'  cuda:{index}: {torch.cuda.get_device_name(index)}')

In [ ]:
import os

result_files = {}
run_environment = os.environ.copy()
run_environment['TOKENIZERS_PARALLELISM'] = 'false'
run_environment['TORCH_HOME'] = '/kaggle/working/torch_home'

for label, checkpoint in checkpoint_paths.items():
    output_dir = OUTPUT_ROOT / label
    result_file = output_dir / 'result/vqa_result.json'
    log_file = output_dir / 'inference.log'
    output_dir.mkdir(parents=True, exist_ok=True)

    if result_file.is_file() and not RERUN_EXISTING:
        print(f'[{label}] Reusing existing result: {result_file}')
        result_files[label] = result_file
        continue

    command = [
        sys.executable,
        '-m',
        'torch.distributed.run',
        '--standalone',
        f'--nproc_per_node={gpu_count}',
        'train_vqa.py',
        f'--config={CONFIG_ARG}',
        f'--output_dir={output_dir}',
        '--evaluate',
        '--no-resume',
        '--overrides',
        f'pretrained={checkpoint}',
    ]
    print(f'[{label}] Evaluating {checkpoint}')
    with log_file.open('w', encoding='utf-8') as log:
        process = subprocess.run(
            command,
            cwd=REPO_DIR,
            env=run_environment,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    if process.returncode != 0:
        log_tail = log_file.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        print('\n'.join(log_tail))
        raise RuntimeError(f'Inference failed for {label}; see {log_file}')
    if not result_file.is_file():
        raise FileNotFoundError(f'Inference finished but result is missing: {result_file}')
    result_files[label] = result_file
    print(f'[{label}] Result: {result_file}')

In [ ]:
metrics_by_checkpoint = {}
annotation_file = pathvqa_root / 'test.json'

for label, result_file in result_files.items():
    command = [
        sys.executable,
        'pathvqa_eval.py',
        str(result_file),
        '--annotation-file',
        str(annotation_file),
    ]
    process = subprocess.run(
        command,
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if process.stdout:
        print(f'[{label}] {process.stdout.strip()}')
    if process.returncode != 0:
        print(process.stderr)
        raise RuntimeError(f'PathVQA scoring failed for {label}')

    metrics_file = result_file.with_name('pathvqa_eval.json')
    if not metrics_file.is_file():
        raise FileNotFoundError(f'Metrics file is missing: {metrics_file}')
    with metrics_file.open(encoding='utf-8') as file:
        metrics_by_checkpoint[label] = json.load(file)

metrics_by_checkpoint

In [ ]:
import pandas as pd

summary = {}
rows = []
for label, metrics in metrics_by_checkpoint.items():
    entry = {
        'checkpoint': label,
        'checkpoint_path': str(checkpoint_paths[label]),
        'result_file': str(result_files[label]),
        'metrics': metrics,
    }
    summary[label] = entry
    rows.append({
        'checkpoint': label,
        'checkpoint_path': str(checkpoint_paths[label]),
        'result_file': str(result_files[label]),
        **metrics,
    })

summary_json = OUTPUT_ROOT / 'pathvqa_eval_summary.json'
summary_csv = OUTPUT_ROOT / 'pathvqa_eval_summary.csv'
with summary_json.open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

summary_df = pd.DataFrame(rows)
if 'overall' in summary_df.columns:
    summary_df = summary_df.sort_values('overall', ascending=False)
summary_df.to_csv(summary_csv, index=False)

display_df = summary_df.copy()
metric_columns = [column for column in display_df.columns if column not in {
    'checkpoint', 'checkpoint_path', 'result_file'
}]
for column in metric_columns:
    display_df[column] = display_df[column].map(
        lambda value: f'{100 * value:.2f}%' if isinstance(value, (int, float)) else value
    )
print(f'JSON summary: {summary_json}')
print(f'CSV summary:  {summary_csv}')
display(display_df)